In [ ]:
import rootutils
import hydra
import yaml

In [ ]:
ROOT = rootutils.setup_root(".", indicator='.project-root', pythonpath=True)

In [ ]:
from omegaconf import OmegaConf
from pathlib import Path
from src.data.combined_dataloader import CombinedAudioDataModule

cfg_path = Path("/mnt/data/kusnierz/repos/SVDD/src/configs/data/combined.yaml")
paths_path = Path("/mnt/data/kusnierz/repos/SVDD/src/configs/paths/default.yaml")

cfg = OmegaConf.load(cfg_path)
paths = OmegaConf.load(paths_path)

merged = OmegaConf.merge(cfg, {"paths": paths})
OmegaConf.resolve(merged)

wild_dir = merged.wild_data_dir
sing_dir = merged.sing_data_dir
batch_size = int(merged.batch_size)
num_workers = int(merged.num_workers)

dataset_kwargs = {}
for k in ("sample_rate", "n_fft", "n_mels", "hop_length", "max_len"):
    if k in merged:
        dataset_kwargs[k] = merged[k]

dm = CombinedAudioDataModule(
    wild_data_dir=wild_dir,
    sing_data_dir=sing_dir,
    batch_size=batch_size,
    num_workers=num_workers,
    **dataset_kwargs
)


In [ ]:
dm.setup(stage="fit")

In [ ]:
train_ds = dm.train_dataset

In [ ]:
len(train_ds)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import Counter
from torch.utils.data import DataLoader
from src.data.base import collate_fn

# zmień jeśli chcesz skanować więcej próbek
SAMPLE_CHECK_N = min(500, len(train_ds))

shapes = []
means = []
stds = []
nan_any = 0
inf_any = 0
labels = []

for i in range(SAMPLE_CHECK_N):
    x, y = train_ds[i]
    shapes.append(tuple(x.shape))
    means.append(float(x.mean()))
    stds.append(float(x.std()))
    nan_any += int(torch.isnan(x).any())
    inf_any += int(torch.isinf(x).any())
    labels.append(int(y.item()))

print("Dataset size:", len(train_ds))
print("Checked samples:", SAMPLE_CHECK_N)
print("Unique shapes (top 10):", Counter(shapes).most_common(10))
print("Label distribution:", Counter(labels))
print("NaN samples (checked subset):", nan_any)
print("Inf samples (checked subset):", inf_any)
print("Mean stats: mean={:.4f}, std(mean)={:.4f}".format(np.mean(means), np.std(means)))
print("Std stats: mean={:.4f}, std(std)={:.4f}".format(np.mean(stds), np.std(stds)))

# Kolory: magma lub viridis działają dobrze dla spektrogramów
def show_spec(x, title=None, vmax=None, vmin=None):
    arr = x.detach().cpu().numpy()
    # usuń wymiar kanału jeśli jest (1, n_mels, T) -> (n_mels, T)
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim == 3:  # (C, M, T) -> take first channel
        arr = arr[0]
    plt.figure(figsize=(10, 3))
    plt.imshow(arr, origin='lower', aspect='auto', cmap='magma', vmax=vmax, vmin=vmin)
    plt.colorbar(format="%.2f")
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()

# pokaż 5 losowych próbek
for idx in random.sample(range(len(train_ds)), min(5, len(train_ds))):
    x, y = train_ds[idx]
    show_spec(x, title=f"idx={idx} label={int(y.item())} shape={tuple(x.shape)}")

# sprawdź działanie collate i batche
dl = DataLoader(train_ds, batch_size=4, collate_fn=collate_fn)
xb, yb, lengths = next(iter(dl))
print("Batch shapes:", xb.shape, yb.shape, "lengths:", lengths)